# Chapter 2: Schema Validation

## The Bouncer That Saves Your Model From Garbage Data

In Chapter 1 we defined what shape data should take (the data contract) and how to translate client columns into standard names (the mapping config). But translation alone doesn't guarantee quality.

What if a client's CSV is missing the `churn_label` column entirely? What if `tenure_months` contains the text `"twelve"` instead of the number `12`? What if the file is empty?

Without a gatekeeper, these problems flow downstream and crash the model (or worse — silently produce garbage predictions).

This chapter introduces the **validator**: a bouncer at the door that checks every dataset against the data contract before it's allowed further into the pipeline.

## How the Bouncer Thinks

The validator's decision logic is dead simple:

1. **Check Tier 1 fields** — are ALL five required fields present?
   - If ANY are missing → **REJECTED** (is_valid = False)
   - If all are present → proceed to step 2

2. **Check types** — do the values make sense for their declared type?
   - `tenure_months` should be numeric, not text
   - `contract_type` should be one of the allowed values
   - Type errors are *reported* but don't cause rejection on their own

3. **Inventory Tier 2/3** — what optional fields are available?
   - These are logged for visibility ("this client gives us 2 of 3 engagement signals")
   - Never cause rejection regardless of how many are missing

The output is a `ValidationResult` that tells you exactly what passed, what failed, and why.

In [1]:
import pandas as pd
from churn_pipeline.steps.validate_data import validate_dataset, ValidationResult
from churn_pipeline.data_contract import STANDARD_SCHEMA, Tier

## Scenario 1: A Perfect Dataset (All Tiers Present)

The best case — client data has everything we could ask for.

In [2]:
# A dataset with ALL fields across all tiers
perfect_data = pd.DataFrame({
    # Tier 1 (Required)
    "customer_id": ["C001", "C002", "C003", "C004", "C005"],
    "tenure_months": [12, 24, 3, 48, 6],
    "monthly_charges": [50.0, 75.0, 95.0, 30.0, 110.0],
    "total_charges": [600.0, 1800.0, 285.0, 1440.0, 660.0],
    "churn_label": [0, 0, 1, 0, 1],
    # Tier 2 (Engagement)
    "contract_type": ["one_year", "two_year", "month-to-month", "two_year", "month-to-month"],
    "payment_method": ["credit_card", "bank_transfer", "electronic_check", "credit_card", "electronic_check"],
    "support_tickets": [0, 1, 5, 0, 3],
    # Tier 3 (Demographics)
    "gender": ["Male", "Female", "Male", "Female", "Male"],
    "partner_status": ["Yes", "No", "No", "Yes", "No"],
})

result = validate_dataset(perfect_data)

print(f"Valid: {result.is_valid}")
print(f"\nTier 1 present: {result.tier1_present}")
print(f"Tier 1 missing: {result.tier1_missing}")
print(f"\nTier 2 present: {result.tier2_present}")
print(f"Tier 2 missing: {result.tier2_missing}")
print(f"\nTier 3 present: {result.tier3_present}")
print(f"Tier 3 missing: {result.tier3_missing}")
print(f"\nErrors: {result.errors}")

Valid: True

Tier 1 present: ['customer_id', 'tenure_months', 'monthly_charges', 'total_charges', 'churn_label']
Tier 1 missing: []

Tier 2 present: ['contract_type', 'payment_method', 'support_tickets']
Tier 2 missing: []

Tier 3 present: ['gender', 'partner_status']
Tier 3 missing: ['age_bucket']

Errors: []


## Scenario 2: Tier 1 Only (Minimum Viable Dataset)

A client sends just the essentials — no engagement signals, no demographics. The bouncer lets them in because all required fields are present. The model will be less accurate, but it can still function.

In [3]:
# Only the bare minimum — Tier 1 fields
minimal_data = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003"],
    "tenure_months": [12, 24, 3],
    "monthly_charges": [50.0, 75.0, 95.0],
    "total_charges": [600.0, 1800.0, 285.0],
    "churn_label": [0, 0, 1],
})

result = validate_dataset(minimal_data)

print(f"Valid: {result.is_valid}  ← Still passes! Tier 1 is complete.")
print(f"\nTier 2 missing: {result.tier2_missing}  ← Logged, not penalized")
print(f"Tier 3 missing: {result.tier3_missing}  ← Logged, not penalized")

Valid: True  ← Still passes! Tier 1 is complete.

Tier 2 missing: ['contract_type', 'payment_method', 'support_tickets']  ← Logged, not penalized
Tier 3 missing: ['gender', 'age_bucket', 'partner_status']  ← Logged, not penalized


## Scenario 3: Missing a Required Field (Rejected)

What happens when the client forgot to include the churn label? Without it, there's nothing to learn from — the model can't train. The bouncer rejects.

In [4]:
# Missing churn_label — can't proceed without the answer column
incomplete_data = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003"],
    "tenure_months": [12, 24, 3],
    "monthly_charges": [50.0, 75.0, 95.0],
    "total_charges": [600.0, 1800.0, 285.0],
    # No churn_label!
})

result = validate_dataset(incomplete_data)

print(f"Valid: {result.is_valid}  ← REJECTED")
print(f"Tier 1 missing: {result.tier1_missing}  ← This is why")

Valid: False  ← REJECTED
Tier 1 missing: ['churn_label']  ← This is why


## Scenario 4: Type Problems (Errors Reported)

Sometimes the column exists, but the values inside are wrong. The IBM Telco dataset has a famous quirk: `TotalCharges` arrives as a string column because some rows contain a blank space `" "` instead of a number.

The validator catches this and reports it — the field IS present (no rejection), but there's a data quality issue to be aware of.

In [5]:
# total_charges has strings that can't convert to float
quirky_data = pd.DataFrame({
    "customer_id": ["C001", "C002", "C003"],
    "tenure_months": [12, 24, 3],
    "monthly_charges": [50.0, 75.0, 95.0],
    "total_charges": ["600.0", " ", "285.0"],  # The " " is a real IBM Telco quirk
    "churn_label": [0, 0, 1],
})

result = validate_dataset(quirky_data)

print(f"Valid: {result.is_valid}  ← Still valid (field exists)")
print(f"Errors: {result.errors}  ← But we know about the data quality issue")

Valid: True  ← Still valid (field exists)
Errors: ["Field 'total_charges': expected float, but 1 values cannot be converted to numeric"]  ← But we know about the data quality issue


## Scenario 5: Empty DataFrame (Edge Case)

What if the file is completely empty? No columns, no rows. This shouldn't crash the validator — it should cleanly report the problem.

In [6]:
empty_data = pd.DataFrame()
result = validate_dataset(empty_data)

print(f"Valid: {result.is_valid}")
print(f"Errors: {result.errors}")
print(f"All Tier 1 missing: {result.tier1_missing}")

Valid: False
Errors: ['DataFrame is empty (0 rows)']
All Tier 1 missing: ['customer_id', 'tenure_months', 'monthly_charges', 'total_charges', 'churn_label']


## The End-to-End Flow: Map Then Validate

In practice, the pipeline does two steps in sequence:
1. Apply the mapping config (translate client columns to standard names)
2. Validate against the data contract (check that the translation produced valid data)

Let's see this full flow with raw client data.

In [7]:
from churn_pipeline.mapping_config import load_mapping_config, apply_mapping

# Raw data as it arrives from the client
raw_telco = pd.DataFrame({
    "customerID": ["7590-VHVEG", "5575-GNVDE", "3668-QPYBK", "9305-CDSKC"],
    "tenure": [1, 34, 2, 45],
    "MonthlyCharges": [29.85, 56.95, 53.85, 42.30],
    "TotalCharges": ["29.85", "1889.5", "108.15", "1840.75"],
    "Churn": ["No", "Yes", "Yes", "No"],
    "Contract": ["Month-to-month", "One year", "Month-to-month", "Two year"],
    "gender": ["Female", "Male", "Male", "Female"],
    "Partner": ["Yes", "No", "No", "Yes"],
})

print("Step 1: Raw data from client")
print(raw_telco.columns.tolist())

# Step 1: Apply mapping
config = load_mapping_config("../configs/client_telco/mapping.yaml")
mapped = apply_mapping(raw_telco, config)

print(f"\nStep 2: After mapping")
print(mapped.columns.tolist())

# Step 2: Validate
result = validate_dataset(mapped)

print(f"\nStep 3: Validation result")
print(f"  Valid: {result.is_valid}")
print(f"  Tier 1 present: {result.tier1_present}")
print(f"  Tier 2 present: {result.tier2_present}")
print(f"  Tier 3 present: {result.tier3_present}")

Step 1: Raw data from client
['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn', 'Contract', 'gender', 'Partner']

Step 2: After mapping
['customer_id', 'tenure_months', 'monthly_charges', 'total_charges', 'churn_label', 'contract_type', 'gender', 'partner_status']

Step 3: Validation result
  Valid: True
  Tier 1 present: ['customer_id', 'tenure_months', 'monthly_charges', 'total_charges', 'churn_label']
  Tier 2 present: ['contract_type']
  Tier 3 present: ['gender', 'partner_status']


## How We Test This: Property-Based Testing

Unit tests check specific examples we thought of. But what about examples we *didn't* think of?

Property-based testing generates hundreds of random inputs and verifies that a universal rule always holds:

- **Property:** For ANY dataset with all Tier 1 fields present → `is_valid` MUST be True
- **Property:** For ANY dataset missing at least one Tier 1 field → `is_valid` MUST be False

We use [Hypothesis](https://hypothesis.readthedocs.io/) to generate 200 random DataFrames per property. If even one violates the rule, the test fails.

In [8]:
# Quick demo: generate a few random valid DataFrames and verify the property holds
import numpy as np

tier1_fields = [name for name, spec in STANDARD_SCHEMA.items() if spec.tier == Tier.REQUIRED]
tier2_fields = [name for name, spec in STANDARD_SCHEMA.items() if spec.tier == Tier.ENGAGEMENT]

print("Generating 10 random DataFrames with all Tier 1 fields...")
for i in range(10):
    n_rows = np.random.randint(1, 50)
    # Pick a random subset of Tier 2 fields to include
    t2_subset = list(np.random.choice(tier2_fields, size=np.random.randint(0, len(tier2_fields)+1), replace=False))
    
    data = {
        "customer_id": [f"C{j}" for j in range(n_rows)],
        "tenure_months": np.random.randint(1, 72, n_rows).tolist(),
        "monthly_charges": np.random.uniform(20, 120, n_rows).tolist(),
        "total_charges": np.random.uniform(100, 8000, n_rows).tolist(),
        "churn_label": np.random.randint(0, 2, n_rows).tolist(),
    }
    if "contract_type" in t2_subset:
        data["contract_type"] = np.random.choice(["month-to-month", "one_year", "two_year"], n_rows).tolist()
    if "payment_method" in t2_subset:
        data["payment_method"] = np.random.choice(["credit_card", "bank_transfer"], n_rows).tolist()
    if "support_tickets" in t2_subset:
        data["support_tickets"] = np.random.randint(0, 10, n_rows).tolist()
    
    df = pd.DataFrame(data)
    result = validate_dataset(df)
    assert result.is_valid, f"Failed on iteration {i}!"
    print(f"  Trial {i+1}: {n_rows} rows, Tier 2 fields={t2_subset} → is_valid={result.is_valid} ✓")

print("\nAll 10 random datasets validated correctly!")

Generating 10 random DataFrames with all Tier 1 fields...
  Trial 1: 3 rows, Tier 2 fields=[np.str_('payment_method')] → is_valid=True ✓
  Trial 2: 37 rows, Tier 2 fields=[] → is_valid=True ✓
  Trial 3: 41 rows, Tier 2 fields=[np.str_('support_tickets'), np.str_('contract_type'), np.str_('payment_method')] → is_valid=True ✓
  Trial 4: 27 rows, Tier 2 fields=[np.str_('contract_type'), np.str_('support_tickets')] → is_valid=True ✓
  Trial 5: 44 rows, Tier 2 fields=[np.str_('support_tickets'), np.str_('payment_method')] → is_valid=True ✓
  Trial 6: 30 rows, Tier 2 fields=[np.str_('contract_type'), np.str_('payment_method')] → is_valid=True ✓
  Trial 7: 37 rows, Tier 2 fields=[] → is_valid=True ✓
  Trial 8: 32 rows, Tier 2 fields=[] → is_valid=True ✓
  Trial 9: 44 rows, Tier 2 fields=[np.str_('support_tickets')] → is_valid=True ✓
  Trial 10: 11 rows, Tier 2 fields=[np.str_('payment_method'), np.str_('contract_type')] → is_valid=True ✓

All 10 random datasets validated correctly!


## Key Takeaways

1. **The validator is a hard gate on Tier 1** — missing required fields = rejected, no exceptions
2. **Tier 2/3 missing is fine** — the model works with less data, just less accurately
3. **Type errors are reported but not blocking** — gives visibility without being overly strict
4. **Map first, then validate** — translation happens before the check
5. **Property-based testing gives confidence** — if the rule holds for 200 random inputs, it's solid

Next: Chapter 3 shows how we transform validated data into numbers the model can actually learn from.

---

*Source code: `src/churn_pipeline/steps/validate_data.py`*  
*Tests: `tests/property/test_schema_validation.py`, `tests/unit/test_validate_data.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*